In [0]:
import pandas as pd

In [0]:
# Load the data from the temporary table created in the gathering notebook
df = spark.table("car_price_gathering_temp")

# Convert to pandas for easier data cleaning
df_pd = df.toPandas()

print(f"Dataset shape: {df_pd.shape}")
print(f"\nColumn names:\n{df_pd.columns.tolist()}")

Dataset shape: (205, 26)

Column names:
['car_ID', 'symboling', 'CarName', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype', 'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg', 'price']


In [0]:
# Check for missing values
print("Missing values per column:")
missing_values = df_pd.isnull().sum()
print(missing_values[missing_values > 0])

print(f"\nTotal missing values: {df_pd.isnull().sum().sum()}")
print(f"Percentage of missing data: {(df_pd.isnull().sum().sum() / (df_pd.shape[0] * df_pd.shape[1])) * 100:.2f}%")

Missing values per column:
Series([], dtype: int64)

Total missing values: 0
Percentage of missing data: 0.00%


In [0]:
# Display basic statistics
print("Numerical columns statistics:")
display(df_pd.describe())

print("\nCategorical columns value counts:")
categorical_cols = df_pd.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n{col}:")
    print(df_pd[col].value_counts())

Numerical columns statistics:


car_ID,symboling,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0,205.0
103.0,0.8341463414634146,98.75658536585367,174.04926829268288,65.90780487804878,53.72487804878049,2555.5658536585365,126.90731707317073,3.329756097560975,3.255414634146341,10.142536585365855,104.1170731707317,5125.121951219512,25.21951219512195,30.75121951219512,13276.710570731706
59.32256456582661,1.2453068281055297,6.021775685025571,12.33728852655518,2.145203852687183,2.4435219699049036,520.6802035016387,41.64269343817984,0.27084370542622926,0.31359701376080407,3.972040321863298,39.54416680936116,476.98564305694634,6.542141653001622,6.886443130941824,7988.85233174315
1.0,-2.0,86.6,141.1,60.3,47.8,1488.0,61.0,2.54,2.07,7.0,48.0,4150.0,13.0,16.0,5118.0
52.0,0.0,94.5,166.3,64.1,52.0,2145.0,97.0,3.15,3.11,8.6,70.0,4800.0,19.0,25.0,7788.0
103.0,1.0,97.0,173.2,65.5,54.1,2414.0,120.0,3.31,3.29,9.0,95.0,5200.0,24.0,30.0,10295.0
154.0,2.0,102.4,183.1,66.9,55.5,2935.0,141.0,3.58,3.41,9.4,116.0,5500.0,30.0,34.0,16503.0
205.0,3.0,120.9,208.1,72.3,59.8,4066.0,326.0,3.94,4.17,23.0,288.0,6600.0,49.0,54.0,45400.0



Categorical columns value counts:

CarName:
CarName
toyota corona           6
toyota corolla          6
peugeot 504             6
subaru dl               4
mitsubishi mirage g4    3
                       ..
mazda glc 4             1
mazda rx2 coupe         1
maxda glc deluxe        1
maxda rx3               1
volvo 246               1
Name: count, Length: 147, dtype: int64

fueltype:
fueltype
gas       185
diesel     20
Name: count, dtype: int64

aspiration:
aspiration
std      168
turbo     37
Name: count, dtype: int64

doornumber:
doornumber
four    115
two      90
Name: count, dtype: int64

carbody:
carbody
sedan          96
hatchback      70
wagon          25
hardtop         8
convertible     6
Name: count, dtype: int64

drivewheel:
drivewheel
fwd    120
rwd     76
4wd      9
Name: count, dtype: int64

enginelocation:
enginelocation
front    202
rear       3
Name: count, dtype: int64

enginetype:
enginetype
ohc      148
ohcf      15
ohcv      13
dohc      12
l         12
rotor   

In [0]:
# Check for duplicate rows
duplicates = df_pd.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    print("\nRemoving duplicate rows...")
    df_pd = df_pd.drop_duplicates()
    print(f"New shape after removing duplicates: {df_pd.shape}")
else:
    print("No duplicates found.")

Number of duplicate rows: 0
No duplicates found.


In [0]:
# Handle missing values
print("Handling missing values...\n")

# For numerical columns: fill with median
numerical_cols = df_pd.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    if df_pd[col].isnull().sum() > 0:
        median_val = df_pd[col].median()
        df_pd[col].fillna(median_val, inplace=True)
        print(f"{col}: Filled {df_pd[col].isnull().sum()} missing values with median ({median_val})")

# For categorical columns: fill with mode
categorical_cols = df_pd.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_pd[col].isnull().sum() > 0:
        mode_val = df_pd[col].mode()[0]
        df_pd[col].fillna(mode_val, inplace=True)
        print(f"{col}: Filled missing values with mode ({mode_val})")

print(f"\nTotal missing values after handling: {df_pd.isnull().sum().sum()}")

Handling missing values...


Total missing values after handling: 0


In [0]:
# Detect and handle outliers using IQR method
import numpy as np

print("Detecting outliers using IQR method...\n")

numerical_cols = df_pd.select_dtypes(include=['int64', 'float64']).columns
# Exclude ID columns from outlier detection
numerical_cols = [col for col in numerical_cols if 'ID' not in col and 'id' not in col]

outlier_counts = {}

for col in numerical_cols:
    Q1 = df_pd[col].quantile(0.25)
    Q3 = df_pd[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define outlier bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count outliers
    outliers = df_pd[(df_pd[col] < lower_bound) | (df_pd[col] > upper_bound)]
    outlier_counts[col] = len(outliers)
    
    if len(outliers) > 0:
        print(f"{col}: {len(outliers)} outliers detected (range: {lower_bound:.2f} - {upper_bound:.2f})")

print(f"\nTotal columns with outliers: {sum(1 for count in outlier_counts.values() if count > 0)}")

Detecting outliers using IQR method...

wheelbase: 3 outliers detected (range: 82.65 - 114.25)
carlength: 1 outliers detected (range: 141.10 - 208.30)
enginesize: 10 outliers detected (range: 31.00 - 207.00)
stroke: 20 outliers detected (range: 2.66 - 3.86)
compressionratio: 28 outliers detected (range: 7.40 - 10.60)
horsepower: 6 outliers detected (range: 1.00 - 185.00)
peakrpm: 2 outliers detected (range: 3750.00 - 6550.00)
citympg: 2 outliers detected (range: 2.50 - 46.50)
highwaympg: 3 outliers detected (range: 11.50 - 47.50)
price: 15 outliers detected (range: -5284.50 - 29575.50)

Total columns with outliers: 10


In [0]:
# Data type validation and conversion
print("Validating and converting data types...\n")

# Convert text numbers to proper format if needed
if 'doornumber' in df_pd.columns:
    door_mapping = {'two': 2, 'four': 4}
    df_pd['doornumber_numeric'] = df_pd['doornumber'].map(door_mapping)
    print(f"Created doornumber_numeric column")

if 'cylindernumber' in df_pd.columns:
    cylinder_mapping = {
        'two': 2, 'three': 3, 'four': 4, 'five': 5, 
        'six': 6, 'eight': 8, 'twelve': 12
    }
    df_pd['cylindernumber_numeric'] = df_pd['cylindernumber'].map(cylinder_mapping)
    print(f"Created cylindernumber_numeric column")

print("\nData type conversion completed.")

Validating and converting data types...

Created doornumber_numeric column
Created cylindernumber_numeric column

Data type conversion completed.


In [0]:
# Final data validation
print("Final data validation...\n")
print(f"Final dataset shape: {df_pd.shape}")
print(f"\nData types:")
print(df_pd.dtypes)
print(f"\nMissing values: {df_pd.isnull().sum().sum()}")
print(f"\nFirst few rows:")
display(df_pd.head())

Final data validation...

Final dataset shape: (205, 28)

Data types:
car_ID                      int64
symboling                   int64
CarName                    object
fueltype                   object
aspiration                 object
doornumber                 object
carbody                    object
drivewheel                 object
enginelocation             object
wheelbase                 float64
carlength                 float64
carwidth                  float64
carheight                 float64
curbweight                  int64
enginetype                 object
cylindernumber             object
enginesize                  int64
fuelsystem                 object
boreratio                 float64
stroke                    float64
compressionratio          float64
horsepower                  int64
peakrpm                     int64
citympg                     int64
highwaympg                  int64
price                     float64
doornumber_numeric          int64
cylindernumb

car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,doornumber_numeric,cylindernumber_numeric
1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,2,4
2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,2,4
3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,2,6
4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.4,10.0,102,5500,24,30,13950.0,4,4
5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.4,8.0,115,5500,18,22,17450.0,4,5


In [0]:
# Convert back to Spark DataFrame and save cleaned data
df_cleaned = spark.createDataFrame(df_pd)

# Save the cleaned data to a new table
cleaned_table_name = "car_price_cleaned"
df_cleaned.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(cleaned_table_name)

print(f"Cleaned data saved to table: {cleaned_table_name}")
print(f"Total rows: {df_cleaned.count()}")
print(f"Total columns: {len(df_cleaned.columns)}")

Cleaned data saved to table: car_price_cleaned
Total rows: 205
Total columns: 28
